In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
df = pd.read_parquet("/content/green_tripdata_2026-01.parquet")
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,4.50,1.5,7.5,0.0,NaN,1.0,45.20,1.0,1.0,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,0.00,0.0,10.2,0.0,NaN,1.0,61.20,1.0,2.0,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,1.00,1.5,2.0,0.0,NaN,1.0,46.00,1.0,1.0,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,3.75,1.5,0.0,0.0,NaN,1.0,25.05,2.0,1.0,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,1.00,0.5,0.0,0.0,NaN,1.0,21.60,2.0,1.0,0.00,0.00


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40272 entries, 0 to 40271
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               40272 non-null  int32         
 1   lpep_pickup_datetime   40272 non-null  datetime64[us]
 2   lpep_dropoff_datetime  40272 non-null  datetime64[us]
 3   store_and_fwd_flag     34858 non-null  object        
 4   RatecodeID             34858 non-null  float64       
 5   PULocationID           40272 non-null  int32         
 6   DOLocationID           40272 non-null  int32         
 7   passenger_count        34858 non-null  float64       
 8   trip_distance          40272 non-null  float64       
 9   fare_amount            40272 non-null  float64       
 10  extra                  40272 non-null  float64       
 11  mta_tax                40272 non-null  float64       
 12  tip_amount             40272 non-null  float64       
 13  t

In [ ]:
df2 = pd.read_parquet("/content/green_tripdata_2025-01.parquet")
df2.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,1.0,0.5,6.80,0.00,NaN,1.0,34.00,1.0,1.0,0.00,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,1.0,0.5,0.00,0.00,NaN,1.0,11.10,2.0,1.0,0.00,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,0.0,0.0,0.00,0.00,NaN,1.0,26.55,2.0,2.0,0.00,0.0
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,N,1.0,74,223,1.0,4.12,21.20,1.0,0.5,6.13,6.94,NaN,1.0,36.77,1.0,1.0,0.00,0.0
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,N,1.0,66,158,1.0,4.71,33.80,1.0,0.5,7.81,0.00,NaN,1.0,46.86,1.0,1.0,2.75,0.0


In [ ]:
from sklearn.feature_extraction import DictVectorizer
# Calculate duration in minutes
df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60
df = df[(df.duration >= 1) & (df.duration <= 60)]
df_features = df[['trip_distance', 'PULocationID', 'DOLocationID']]
dict_vect = DictVectorizer(sparse=True)
X = dict_vect.fit_transform(df_features.to_dict(orient='records'))
y = df.duration.values

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((30470, 3), (7618, 3), (30470,), (7618,))

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

# Make predictions
y_pred = lr.predict(X_val)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"Linear Regression RMSE: {rmse:.2f} minutes")
print(f"Mean Absolute Error Score: {mae:.2f}")

Linear Regression RMSE: 10.21 minutes
Mean Absolute Error Score: 7.52


In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
print(f"Random Forest RMSE: {rmse:.2f} minutes")
print(f"Mean Absolute Error Score: {mae:.2f}")

Random Forest RMSE: 6.20 minutes
Mean Absolute Error Score: 4.19


In [ ]:
with open("trained_model.pkl", "wb") as file:
    pickle.dump(rf, file)